# Data preprocessing

In [2]:
from dotenv import load_dotenv
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

load_dotenv()

dataset_url = os.getenv("DATASET_URL")

train_dataset = tf.keras.utils.image_dataset_from_directory(
  dataset_url,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=(180,180),
  batch_size=16)

val_dataset = tf.keras.utils.image_dataset_from_directory(
  dataset_url,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=(180,180),
  batch_size=16)

AUTOTUNE = tf.data.AUTOTUNE

train_dataset_nm = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_dataset_nm = val_dataset.cache().prefetch(buffer_size=AUTOTUNE)


normalization_layer = layers.Rescaling(1./255)

normalized_train_dataset_nm = train_dataset_nm.map(lambda x, y: (normalization_layer(x), y))
normalized_val_dataset_nm = val_dataset_nm.map(lambda x, y: (normalization_layer(x), y))


Found 4000 files belonging to 2 classes.
Using 3200 files for training.
Found 4000 files belonging to 2 classes.
Using 800 files for validation.


# Building the CNN model

In [3]:
from tensorflow.keras.models import Sequential

data_augmentation = keras.Sequential([
  layers.RandomFlip("horizontal", input_shape=(180, 180, 3)),
  layers.RandomRotation(0.1),
  layers.RandomZoom(0.1),
])

model = Sequential([
  data_augmentation,
  layers.Rescaling(1./255),
  layers.Conv2D(16,3,padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(32,3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(64,3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Dropout(0.2),
  layers.Flatten(),
  layers.Dropout(0.5),
  layers.Dense(128, activation='relu'),
  layers.Dense(len(train_dataset.class_names))
])

model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])



c:\Users\vemir\DeepLeaf\Bikes-Vs-Cars-CNN-Model\tf-env\Lib\site-packages\keras\src\layers\preprocessing\data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


# Model Training

In [4]:
epochs = 5

history = model.fit(
  normalized_train_dataset_nm,
  validation_data=normalized_val_dataset_nm,
  epochs=epochs
)

Epoch 1/5


c:\Users\vemir\DeepLeaf\Bikes-Vs-Cars-CNN-Model\tf-env\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


200/200 ━━━━━━━━━━━━━━━━━━━━ 37s 157ms/step - accuracy: 0.4975 - loss: 0.6934 - val_accuracy: 0.5125 - val_loss: 0.6930
Epoch 2/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 31s 155ms/step - accuracy: 0.4975 - loss: 0.6933 - val_accuracy: 0.4875 - val_loss: 0.6932
Epoch 3/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 29s 146ms/step - accuracy: 0.4888 - loss: 0.6932 - val_accuracy: 0.4875 - val_loss: 0.6932
Epoch 4/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 30s 148ms/step - accuracy: 0.4963 - loss: 0.6933 - val_accuracy: 0.4875 - val_loss: 0.6932
Epoch 5/5
200/200 ━━━━━━━━━━━━━━━━━━━━ 31s 154ms/step - accuracy: 0.4888 - loss: 0.6932 - val_accuracy: 0.4875 - val_loss: 0.6933


# Model Evaluation